<a href="https://colab.research.google.com/github/lovajujo/sport-data/blob/main/worldcup_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
pip install requests beautifulsoup4 pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 2.2 MB/s eta 0:00:00


In [ ]:
# @title
import csv
import io
import re
import time
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader

# --- PHASE 1: DATA EXTRACTION ---
LISTING_URL = "https://www.fifatrainingcentre.com/en/fifa-world-cup-2026/match-report-hub.php"
ROOT_DOMAIN = "https://www.fifatrainingcentre.com"
scraped_data = []
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}

print("Connecting to FIFA Match Report Hub...")
try:
    response = requests.get(LISTING_URL, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    pdf_urls = []
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        if 'pmsr-' in href.lower() and href.lower().endswith('.pdf'):
            full_url = href if href.startswith('http') else f"{ROOT_DOMAIN}{href if href.startswith('/') else '/' + href}"
            if full_url not in pdf_urls: pdf_urls.append(full_url)
except Exception as e:
    print(f"❌ Error setting up extraction: {e}")
    exit()

print(f"Found {len(pdf_urls)} match reports. Streaming to memory...")
for url in pdf_urls:
    try:
        pdf_response = requests.get(url, headers=headers)
        if pdf_response.status_code != 200 or len(pdf_response.content) < 100: continue
        reader = PdfReader(io.BytesIO(pdf_response.content))
        combined_text = "".join([reader.pages[i].extract_text() or "" for i in range(min(3, len(reader.pages)))])

        # Regex parsers matching the layout
        teams_match = re.search(r"Match Summary\s*[-–—\s]*Key\s+Statistics\s*\.?\s*([^.\n]+)\s*\.\s*([^.\n]+)\s*\.\s*Possession", combined_text, re.IGNORECASE)
        goals_match = re.search(r"(\d+)\s*\.?\s*Goals\s*\.?\s*(\d+)", combined_text)
        xg_match = re.search(r"(\d+\.\d+)\s*\.?\s*xG\s*\(Expected\s*Goals\)\s*\.?\s*(\d+\.\d+)", combined_text, re.IGNORECASE)

        if teams_match and goals_match and xg_match:
            home_team = teams_match.group(1).strip()
            away_team = teams_match.group(2).strip()
            hg, ag = map(int, goals_match.groups())
            h_xg, a_xg = map(float, xg_match.groups())

            scraped_data.append({
                "home": home_team, "away": away_team,
                "hg": hg, "ag": ag, "h_xg": h_xg, "a_xg": a_xg
            })
        time.sleep(0.2)
    except Exception:
        continue

if not scraped_data:
    print("❌ No data could be processed. Please verify the URL data layer access.")
    exit()





Connecting to FIFA Match Report Hub...
Found 72 match reports. Streaming to memory...


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# 1. Replace with your modified sheet URL
sheet_url = "https://docs.google.com/spreadsheets/d/1v2gWp7lBC6oHfEQZ04spdHqpkBjIwo0Xm7utgU7Rilc/export?format=csv"

# 2. Load directly into a DataFrame
df = pd.read_csv(sheet_url)

# View the match data layer
print(df.tail())

          Date Home Away  Home Goals  Away Goals  Home xG  Away xG
95  2026.07.07  SUI  COL           0           0     0.58     0.97
96  2026.07.09  FRA  MAR           2           0     3.52     0.16
97  2026.07.10  ESP  BEL           2           1     2.20     0.34
98  2026.07.11  NOR  ENG           1           2     0.87     1.64
99  2026.07.11  ARG  SUI           3           1     1.99     0.44


In [ ]:
import numpy as np
# =========================================================
# PHASE 2: TEAM PROFILE METRICS ALGORITHM (PANDAS VERSION)
# =========================================================
df.columns = df.columns.str.strip()
df['Home'] = df['Home'].astype(str).str.strip()
df['Away'] = df['Away'].astype(str).str.strip()

# 1. Determine points for every match row
df['home_pts'] = np.where(df['Home Goals'] > df['Away Goals'], 3,
                 np.where(df['Home Goals'] == df['Away Goals'], 1, 0))

df['away_pts'] = np.where(df['Away Goals'] > df['Home Goals'], 3,
                 np.where(df['Home Goals'] == df['Away Goals'], 1, 0))

# 2. Aggregate stats from the Home perspective
home_stats = df.groupby('Home').agg(
    gp=('Home', 'count'),
    pts=('home_pts', 'sum'),
    gf=('Home Goals', 'sum'),
    ga=('Away Goals', 'sum'),
    xg=('Home xG', 'sum'),
    xga=('Away xG', 'sum')
).rename_axis('Team')

# 3. Aggregate stats from the Away perspective
away_stats = df.groupby('Away').agg(
    gp=('Away', 'count'),
    pts=('away_pts', 'sum'),
    gf=('Away Goals', 'sum'),
    ga=('Home Goals', 'sum'),
    xg=('Away xG', 'sum'),
    xga=('Home xG', 'sum')
).rename_axis('Team')

# 4. Combine Home and Away stats into a single master Team Profiles matrix
# .add(..., fill_value=0) handles teams that might have only played home or away games so far
team_profiles = home_stats.add(away_stats, fill_value=0)

# 5. Run your core metric calculations across the whole table at once
team_profiles['ppg'] = team_profiles['pts'] / team_profiles['gp']
team_profiles['avg_gf'] = team_profiles['gf'] / team_profiles['gp']
team_profiles['avg_ga'] = team_profiles['ga'] / team_profiles['gp']
team_profiles['avg_xg'] = team_profiles['xg'] / team_profiles['gp']
team_profiles['avg_xga'] = team_profiles['xga'] / team_profiles['gp']

# 6. Calculate Attacking and Defending Efficiency
team_profiles['att_eff'] = (team_profiles['gf']+1) / (team_profiles['xg']+1)
team_profiles['def_eff'] = (team_profiles['ga'] +1)/ (team_profiles['xga']+1)

# 7. Baseline baseline baseline average xG across all match environments
# (Total xG created by all teams divided by total team match instances)
all_team_avg_xg = team_profiles['xg'].sum() / team_profiles['gp'].sum()

# 1. Force the index to be treated strictly as clean, native Python strings
team_profiles.index = team_profiles.index.astype(str).str.strip()

# 2. Re-index the internal Pandas table to drop any hidden multi-index levels
team_profiles = team_profiles.loc[~team_profiles.index.duplicated(keep='first')]

# 3. Clean up any trailing structural artifacts in the index name
team_profiles.index.name = 'Team'

print("✅ Phase 2 Complete: Team profiles computed successfully from DataFrame.")
print(team_profiles.head())

✅ Phase 2 Complete: Team profiles computed successfully from DataFrame.
       gp   pts    gf   ga     xg   xga       ppg    avg_gf    avg_ga  \
Team                                                                    
ALG   4.0   4.0   5.0  9.0   4.73  4.92  1.000000  1.250000  2.250000   
ARG   6.0  18.0  17.0  6.0  13.90  3.72  3.000000  2.833333  1.000000   
AUS   4.0   5.0   3.0  3.0   2.74  4.76  1.250000  0.750000  0.750000   
AUT   4.0   4.0   6.0  9.0   3.87  6.77  1.000000  1.500000  2.250000   
BEL   6.0  11.0  14.0  7.0  10.61  6.13  1.833333  2.333333  1.166667   

        avg_xg   avg_xga   att_eff   def_eff  
Team                                          
ALG   1.182500  1.230000  1.047120  1.689189  
ARG   2.316667  0.620000  1.208054  1.483051  
AUS   0.685000  1.190000  1.069519  0.694444  
AUT   0.967500  1.692500  1.437372  1.287001  
BEL   1.768333  1.021667  1.291990  1.122020  


In [ ]:
def predict_match(home, away):
    home = str(home).strip()
    away = str(away).strip()

    if home not in team_profiles.index or away not in team_profiles.index:
        return None

    h_prof = team_profiles.loc[home]
    a_prof = team_profiles.loc[away]

    # 1. Apply Laplace Smoothing to Efficiency right here to keep data clean
    # (Adds 1 phantom goal / 1 xG baseline)
    h_att = (h_prof['gf'] + 1.0) / (h_prof['xg'] + 1.0)
    h_def = (h_prof['ga'] + 1.0) / (h_prof['xga'] + 1.0)
    a_att = (a_prof['gf'] + 1.0) / (a_prof['xg'] + 1.0)
    a_def = (a_prof['ga'] + 1.0) / (a_prof['xga'] + 1.0)

    # 2. Calculate the PPG form modifiers
    h_ppg_mod = 1 + (h_prof['ppg'] - a_prof['ppg'])
    a_ppg_mod = 1 + (a_prof['ppg'] - h_prof['ppg'])

    # FIX: Prevent the PPG multiplier from dropping below 0.20
    # This stops strong teams from completely wiping out an opponent's score to absolute zero
    h_ppg_mod = max(0.20, h_ppg_mod)
    a_ppg_mod = max(0.20, a_ppg_mod)

    # 3. Your Exact Core Formula Layout
    # Home attack vs Away defense
    pred_home = h_prof['avg_xg']/all_team_avg_xg * a_prof['avg_xga']/all_team_avg_xg * h_prof['att_eff'] * a_prof['def_eff']


    # Away attack vs Home defense
    pred_away = a_prof['avg_xg']/all_team_avg_xg * h_prof['avg_xga']/all_team_avg_xg * a_prof['att_eff'] * h_prof['def_eff']

    # Final floor check
    pred_home = max(0.0, pred_home)
    pred_away = max(0.0, pred_away)

    print(f"\n Game: {home} vs {away}")
    print(f"  -> Predicted xG: {pred_home:.2f} - {pred_away:.2f}")

    return pred_home, pred_away

In [ ]:
print(predict_match("FRA", "ESP"),
      predict_match("ENG", "ARG"))



 Game: FRA vs ESP
  -> Predicted xG: 0.36 - 0.45

 Game: ENG vs ARG
  -> Predicted xG: 1.24 - 1.72
(np.float64(0.3637438242532504), np.float64(0.4521146093869011)) (np.float64(1.240180144874278), np.float64(1.7226776018246173))
